# Код для сайта-агрегатора "Отзовик"

In [ ]:
import time
import csv
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException, ElementClickInterceptedException

START_URL = "https://otzovik.com/reviews/on-layn_shkola_nutriciologii_pro-zdorove/"
CSV_FILE = "otzovik_prozdorove_reviews.csv"
MAX_PAGES = 179
DELAY = 2
PAGE_LOAD_DELAY = 5

options = webdriver.ChromeOptions()
options.add_argument("--disable-blink-features=automation")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
driver = webdriver.Chrome(options=options)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")


def format_date(date_str):
    # на сайте даты в духе "23 ноя 2023", приводим к 23.11.2023
    if '.' in date_str:
        return date_str

    months = {
        'янв': '01', 'фев': '02', 'мар': '03', 'апр': '04',
        'май': '05', 'июн': '06', 'июл': '07', 'авг': '08',
        'сен': '09', 'окт': '10', 'ноя': '11', 'дек': '12',
        'января': '01', 'февраля': '02', 'марта': '03', 'апреля': '04',
        'мая': '05', 'июня': '06', 'июля': '07', 'августа': '08',
        'сентября': '09', 'октября': '10', 'ноября': '11', 'декабря': '12'
    }

    parts = date_str.split()
    if len(parts) == 3:
        day, month_word, year = parts
        day = day.rstrip('.')
        month_word = month_word.lower().rstrip('.')
        if month_word in months:
            day = day.zfill(2)
            month = months[month_word]
            return f"{day}.{month}.{year}"

    return date_str


def load_existing_reviews(filename, url_column='url_review'):
    # чтобы не качать одно и то же дважды при повторных запусках
    existing_urls = set()
    if os.path.exists(filename):
        with open(filename, 'r', encoding='utf-8-sig') as f:
            reader = csv.DictReader(f)
            fieldnames = reader.fieldnames
            if url_column not in fieldnames and 'URL' in fieldnames:
                url_column = 'URL'
            for row in reader:
                if url_column in row and row[url_column]:
                    existing_urls.add(row[url_column].strip())
    return existing_urls


def get_current_max_id(filename, id_column='review_id'):
    if not os.path.exists(filename):
        return 0
    max_id = 0
    with open(filename, 'r', encoding='utf-8-sig') as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                val = int(row[id_column])
                if val > max_id:
                    max_id = val
            except (ValueError, KeyError):
                continue
    return max_id


def prepare_csv_file(filename, expected_headers):
    # если когда-то писали другую шапку — отодвигаем старый файл в бэкап, чтобы не ломать структуру
    if not os.path.exists(filename):
        return
    with open(filename, 'r', encoding='utf-8-sig') as f:
        reader = csv.reader(f)
        try:
            first_row = next(reader)
        except StopIteration:
            os.remove(filename)
            return
    if first_row != expected_headers:
        backup_name = f"{filename}.backup_{int(time.time())}"
        os.rename(filename, backup_name)
        print(f"Старый файл переименован в {backup_name}. Будет создан новый файл с правильной структурой.")


def save_review_to_csv(review_id, username, review_text, review_date, review_url, filename):
    # схлопываем переносы строк, иначе CSV потом превращается в кашу
    review_text = ' '.join(review_text.splitlines())
    review_text = ' '.join(review_text.split())

    file_exists = os.path.exists(filename)
    with open(filename, 'a', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f, delimiter=',', quotechar='"',
                            quoting=csv.QUOTE_ALL, lineterminator='\n')
        if not file_exists:
            writer.writerow(['review_id', 'user_id', 'comment', 'date', 'url_review'])
        writer.writerow([review_id, username, review_text, review_date, review_url])


def scrape_page(driver, existing_urls, start_id, need_count):
    collected = 0
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.item"))
        )
    except TimeoutException:
        print("Не найдены отзывы на странице.")
        return 0

    review_blocks = driver.find_elements(By.CSS_SELECTOR, "div.item")
    print(f"Найдено отзывов на странице: {len(review_blocks)}")

    main_window = driver.current_window_handle

    for idx, block in enumerate(review_blocks, 1):
        if collected >= need_count:
            print(f"   Достигнут лимит сбора на этой странице, обработано {collected} отзывов.")
            break

        try:
            username_elem = block.find_element(By.CSS_SELECTOR, "a.user-login span")
            username = username_elem.text.strip()
        except NoSuchElementException:
            username = "Неизвестно"

        try:
            date_elem = block.find_element(By.CSS_SELECTOR, "div.review-postdate span")
            raw_date = date_elem.text.strip()
            review_date = format_date(raw_date)
        except NoSuchElementException:
            review_date = "Неизвестно"

        try:
            read_link = block.find_element(By.CSS_SELECTOR, "a.review-btn.review-read-link")
            review_url = read_link.get_attribute("href")
        except NoSuchElementException:
            print(f"   Пропускаем блок #{idx}: нет ссылки на отзыв")
            continue

        if review_url in existing_urls:
            print(f"   Отзыв {review_url} уже есть, пропускаем")
            continue

        # на превью текст обрезан, поэтому идём на страницу самого отзыва
        driver.execute_script("window.open(arguments[0]);", review_url)
        time.sleep(DELAY)
        driver.switch_to.window(driver.window_handles[-1])
        time.sleep(PAGE_LOAD_DELAY)

        full_text = ""
        try:
            text_elem = driver.find_element(By.CSS_SELECTOR, "div.review-body")
            full_text = text_elem.text.strip()
        except NoSuchElementException:
            # запасной селектор — попадался на старых отзывах
            try:
                text_elem = driver.find_element(By.CSS_SELECTOR, "div.review-full-text")
                full_text = text_elem.text.strip()
            except NoSuchElementException:
                print(f"   Не удалось найти текст отзыва на странице {review_url}")

        if full_text:
            current_id = start_id + collected
            save_review_to_csv(current_id, username, full_text, review_date, review_url, CSV_FILE)
            existing_urls.add(review_url)
            collected += 1
            print(f"   Собран отзыв #{current_id}: {username} ({review_date})")
        else:
            print(f"   Не удалось получить текст для {review_url}")

        driver.close()
        driver.switch_to.window(main_window)
        time.sleep(DELAY)

    return collected


def go_to_next_page(driver):
    try:
        next_btn = driver.find_element(By.XPATH, "//a[.//span[contains(text(),'Далее')]]")
        driver.execute_script("arguments[0].scrollIntoView();", next_btn)
        time.sleep(1)
        next_btn.click()
        return True
    except NoSuchElementException:
        # вёрстка пагинации периодически меняется, пробуем альтернативные классы
        try:
            next_btn = driver.find_element(By.CSS_SELECTOR, "a.pager-next, a.next")
            driver.execute_script("arguments[0].scrollIntoView();", next_btn)
            time.sleep(1)
            next_btn.click()
            return True
        except NoSuchElementException:
            print("Кнопка 'Далее' не найдена. Вероятно, это последняя страница.")
            return False
    except ElementClickInterceptedException:
        # иногда поверх кнопки висит баннер — кликаем через JS
        driver.execute_script("arguments[0].click();", next_btn)
        return True


def main():
    expected_headers = ['review_id', 'user_id', 'comment', 'date', 'url_review']
    prepare_csv_file(CSV_FILE, expected_headers)

    current_max_id = get_current_max_id(CSV_FILE)
    next_id = current_max_id + 1
    print(f"Текущий максимальный ID: {current_max_id}. Следующий ID: {next_id}")

    target_input = input("Сколько отзывов собрать? (по умолчанию 300): ").strip()
    if not target_input:
        target_count = 300
    else:
        target_count = int(target_input)

    existing_urls = load_existing_reviews(CSV_FILE, 'url_review')
    print(f"Загружено {len(existing_urls)} уже собранных отзывов.")

    driver.get(START_URL)
    time.sleep(PAGE_LOAD_DELAY)

    # отзовик любит подсовывать капчу — сидим и ждём, пока человек её решит руками
    if "captcha" in driver.current_url or "Вы робот?" in driver.title:
        print("Обнаружена капча. Пожалуйста, вручную введите код и нажмите 'Я не робот'.")
        print("Скрипт ожидает, пока капча не будет пройдена...")
        while True:
            if "captcha" not in driver.current_url and "Вы робот?" not in driver.title:
                try:
                    driver.find_element(By.CSS_SELECTOR, "form.captcha-form")
                    time.sleep(1)
                    continue
                except NoSuchElementException:
                    break
            time.sleep(2)
        print("Капча пройдена. Ожидаем загрузки отзывов...")
        time.sleep(3)
        try:
            WebDriverWait(driver, 30).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "div.item"))
            )
            print("Отзывы загружены, продолжаем сбор.")
        except TimeoutException:
            # бывает, что после капчи кидает на главную — возвращаемся на нужную страницу
            print("Не удалось дождаться загрузки отзывов после капчи. Пробуем перейти по START_URL.")
            driver.get(START_URL)
            time.sleep(PAGE_LOAD_DELAY)
            try:
                WebDriverWait(driver, 30).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "div.item"))
                )
                print("Отзывы загружены после перехода.")
            except TimeoutException:
                print("Отзывы не найдены. Возможно, сайт изменил структуру. Выход.")
                driver.quit()
                return

    page_num = 1
    new_collected = 0

    while new_collected < target_count:
        print(f"\n--- Обрабатывается страница {page_num} ---")
        need = target_count - new_collected
        collected_on_page = scrape_page(driver, existing_urls, next_id, need)

        new_collected += collected_on_page
        next_id += collected_on_page
        print(f"Собрано на странице: {collected_on_page}, всего за сессию: {new_collected}")

        if new_collected >= target_count:
            print(f"Собрано {new_collected} отзывов, цель достигнута.")
            break

        if page_num >= MAX_PAGES:
            print(f"Достигнут лимит страниц ({MAX_PAGES}). Завершаем.")
            break

        if not go_to_next_page(driver):
            print("Больше страниц нет. Завершаем сбор.")
            break

        page_num += 1
        time.sleep(PAGE_LOAD_DELAY)

    print(f"\nСбор завершён. Всего новых отзывов добавлено: {new_collected}. Всего в файле теперь: {len(existing_urls)}")
    driver.quit()


if __name__ == "__main__":
    main()

# Код для сайта-агрегатора "Сравни.ру"

In [ ]:
import datetime
import time
import csv
import os
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException

START_URL = "https://www.sravni.ru/shkola/edpro/beauty/otzyvy/?filterby=all"
CSV_FILE = "sravni_edpro_reviews_beauty_health.csv"
PAGE_LOAD_DELAY = 5
SCROLL_PAUSE_TIME = 8
MAX_SCROLL_ATTEMPTS = 30

options = webdriver.ChromeOptions()
options.add_argument("--disable-blink-features=automation")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")


def clean_review_text(text):
    # сравни любит дописывать в конец "Читать"/"Скрыть" и метку "клиент Сравни" — выпиливаем
    if not text:
        return text
    text = re.sub(r'\s*клиент\s+сравни\s*', ' ', text, flags=re.IGNORECASE)
    # "срыть" — это не опечатка с моей стороны, такое реально встречалось в разметке
    text = re.sub(r'\s*(Читать|Скрыть|срыть)[\s\.!?]*$', '', text, flags=re.IGNORECASE)
    text = ' '.join(text.split())
    return text.strip()


def format_date(date_str):
    months = {
        'янв': '01', 'фев': '02', 'мар': '03', 'апр': '04',
        'май': '05', 'июн': '06', 'июл': '07', 'авг': '08',
        'сен': '09', 'окт': '10', 'ноя': '11', 'дек': '12',
        'января': '01', 'февраля': '02', 'марта': '03', 'апреля': '04',
        'мая': '05', 'июня': '06', 'июля': '07', 'августа': '08',
        'сентября': '09', 'октября': '10', 'ноября': '11', 'декабря': '12'
    }
    date_str = date_str.lstrip(', ').strip()
    parts = date_str.split()
    if len(parts) == 3:
        day, month_word, year = parts
        day = day.rstrip('.')
        month_word = month_word.lower().rstrip('.')
        if month_word in months:
            day = day.zfill(2)
            month = months[month_word]
            return f"{day}.{month}.{year}"
    elif len(parts) == 2:
        # свежие отзывы идут без года, типа "16 февраля" — подставляем текущий
        day, month_word = parts
        day = day.rstrip('.')
        month_word = month_word.lower().rstrip('.')
        if month_word in months:
            day = day.zfill(2)
            month = months[month_word]
            current_year = str(datetime.datetime.now().year)
            return f"{day}.{month}.{current_year}"
    return date_str


def load_existing_reviews(filename):
    existing_urls = set()
    if os.path.exists(filename):
        with open(filename, 'r', encoding='utf-8-sig') as f:
            reader = csv.DictReader(f)
            for row in reader:
                if 'url_review' in row and row['url_review']:
                    existing_urls.add(row['url_review'].strip())
    return existing_urls


def get_current_max_id(filename):
    if not os.path.exists(filename):
        return 0
    max_id = 0
    with open(filename, 'r', encoding='utf-8-sig') as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                val = int(row['review_id'])
                if val > max_id:
                    max_id = val
            except (ValueError, KeyError):
                continue
    return max_id


def prepare_csv_file(filename):
    expected_headers = ['review_id', 'user_id', 'comment', 'date', 'url_review']
    if not os.path.exists(filename):
        return
    with open(filename, 'r', encoding='utf-8-sig') as f:
        reader = csv.reader(f)
        try:
            first_row = next(reader)
        except StopIteration:
            os.remove(filename)
            return
    if first_row != expected_headers:
        backup_name = f"{filename}.backup_{int(time.time())}"
        os.rename(filename, backup_name)
        print(f"Старый файл переименован в {backup_name}")


def save_review_to_csv(review_id, username, review_text, review_date, review_url, filename):
    review_text = ' '.join(review_text.splitlines())
    review_text = ' '.join(review_text.split())
    file_exists = os.path.exists(filename)
    with open(filename, 'a', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f, delimiter=',', quotechar='"',
                            quoting=csv.QUOTE_ALL, lineterminator='\n')
        if not file_exists:
            writer.writerow(['review_id', 'user_id', 'comment', 'date', 'url_review'])
        writer.writerow([review_id, username, review_text, review_date, review_url])


def slow_scroll_to(driver, element, step=80, delay=0.8):
    # резкий scrollIntoView пугает ленивую подгрузку — крутим по чуть-чуть
    target_y = driver.execute_script("return arguments[0].getBoundingClientRect().top + window.pageYOffset;", element)
    current_y = driver.execute_script("return window.pageYOffset;")
    distance = target_y - current_y
    if distance > 0:
        steps = max(1, int(distance / step) + 1)
        for i in range(steps):
            scroll_by = min(step, distance - i*step)
            if scroll_by <= 0:
                break
            driver.execute_script(f"window.scrollBy(0, {scroll_by});")
            time.sleep(delay)
    elif distance < 0:
        steps = max(1, int(abs(distance) / step) + 1)
        for i in range(steps):
            scroll_by = -min(step, abs(distance) - i*step)
            if scroll_by >= 0:
                break
            driver.execute_script(f"window.scrollBy(0, {scroll_by});")
            time.sleep(delay)


def load_all_reviews(driver):
    # пагинации нет, всё на бесконечном скролле — крутим до упора
    review_selector = "div.review-card_wrapper__s9I1u"
    for attempt in range(MAX_SCROLL_ATTEMPTS):
        current_reviews = driver.find_elements(By.CSS_SELECTOR, review_selector)
        current_count = len(current_reviews)
        print(f"Загружено отзывов: {current_count}")

        if current_count == 0:
            print("Отзывы не найдены.")
            break

        last_review = current_reviews[-1]
        slow_scroll_to(driver, last_review, step=80, delay=0.8)
        time.sleep(1.5)

        # признак конца списка — после прокрутки число карточек не выросло
        try:
            WebDriverWait(driver, SCROLL_PAUSE_TIME).until(
                lambda d: len(d.find_elements(By.CSS_SELECTOR, review_selector)) > current_count
            )
            print("Новые отзывы загрузились.")
        except TimeoutException:
            print("Новые отзывы не загрузились. Достигнут конец списка.")
            break

    return driver.find_elements(By.CSS_SELECTOR, review_selector)


def scrape_page(driver, existing_urls, start_id, need_count):
    collected = 0

    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.review-card_wrapper__s9I1u"))
        )
    except TimeoutException:
        print("Не найдены отзывы на странице.")
        return 0

    print("Загружаем все отзывы...")
    parent_cards = load_all_reviews(driver)
    print(f"Всего загружено отзывов: {len(parent_cards)}")

    for idx, card in enumerate(parent_cards, 1):
        if collected >= need_count:
            print(f"Достигнут лимит сбора ({need_count})")
            break

        print(f"\n--- Обработка отзыва #{idx} ---")

        try:
            username_elem = card.find_element(By.CSS_SELECTOR, "div.h-color-D100")
            username = username_elem.text.strip()
        except NoSuchElementException:
            username = "Неизвестно"
        print(f"   Имя: {username}")

        review_date = "Неизвестно"
        raw_date = ""
        try:
            date_elem = card.find_element(By.CSS_SELECTOR, "div.h-ml-12 div.h-color-D30")
            raw_date = date_elem.text.strip()
            review_date = format_date(raw_date)
        except NoSuchElementException:
            # у части карточек обёртки h-ml-12 нет, пробуем без неё
            try:
                date_elem = card.find_element(By.CSS_SELECTOR, "div.h-color-D30")
                raw_date = date_elem.text.strip()
                review_date = format_date(raw_date)
            except NoSuchElementException:
                pass
        print(f"   Дата: {review_date}")

        try:
            link_elem = card.find_element(By.CSS_SELECTOR, "a.review-card_link__zq9dQ")
            review_url = link_elem.get_attribute("href")
        except NoSuchElementException:
            # отдельной ссылки на отзыв может не быть — лепим якорь, чтобы хоть как-то дедуплицировать
            review_url = f"{START_URL}#review-{idx}"

        if review_url in existing_urls:
            print(f"   Отзыв {review_url} уже есть, пропускаем")
            continue

        # длинные отзывы свёрнуты, надо тыкнуть "Читать" — иначе утащим обрезанный текст
        try:
            read_more_btn = card.find_element(By.CSS_SELECTOR, "a.h-ml-4, a[class*='_i91ye']")
            driver.execute_script("arguments[0].scrollIntoView();", read_more_btn)
            time.sleep(0.5)

            # запоминаем длину до клика, чтобы потом дождаться реального раскрытия
            try:
                text_block = card.find_element(By.CSS_SELECTOR, "div.review-card_text__a8E3W")
                text_before_len = len(text_block.text)
            except NoSuchElementException:
                text_before_len = 0

            driver.execute_script("arguments[0].click();", read_more_btn)
            print("   Кнопка 'Читать' нажата")

            WebDriverWait(driver, 10).until(
                lambda d: len(d.find_element(By.CSS_SELECTOR, "div.review-card_text__a8E3W").text) > text_before_len
            )
            time.sleep(1)
        except NoSuchElementException:
            print("   Кнопка 'Читать' не найдена (текст уже полный)")
        except TimeoutException:
            print("   Не дождались увеличения текста после клика, но пробуем извлечь, что есть")

        full_text = ""
        try:
            text_block = card.find_element(By.CSS_SELECTOR, "div.review-card_text__a8E3W")
            # innerText вместо .text — забирает списки, цитаты и переносы как видит юзер
            full_text = driver.execute_script("return arguments[0].innerText;", text_block).strip()

            # бывает, кнопка "Читать" остаётся внутри блока текстом
            if full_text.endswith("Читать"):
                full_text = full_text[:-6].strip()

            print(f"   Текст (первые 100 символов): {full_text[:100]}")
        except NoSuchElementException:
            # фоллбэк на случай, если разметку поменяли: берём весь card.text и руками отрезаем шапку
            print("   Блок review-card_text не найден, берём текст всей карточки")
            full_text = card.text.strip()
            if username and username != "Неизвестно" and username in full_text:
                full_text = full_text.replace(username, "", 1).strip()
            if raw_date and raw_date in full_text:
                full_text = full_text.replace(raw_date, "", 1).strip()
            if full_text.endswith("Читать"):
                full_text = full_text[:-6].strip()
        except Exception as e:
            print(f"   Ошибка при получении текста: {e}")
            continue

        full_text = clean_review_text(full_text)

        if full_text:
            current_id = start_id + collected
            save_review_to_csv(current_id, username, full_text, review_date, review_url, CSV_FILE)
            existing_urls.add(review_url)
            collected += 1
            print(f"   Собран отзыв #{current_id}: {username} ({review_date})")
        else:
            print(f"   Пустой текст после очистки, отзыв НЕ сохранён")

    return collected


def main():
    prepare_csv_file(CSV_FILE)
    current_max_id = get_current_max_id(CSV_FILE)
    next_id = current_max_id + 1
    print(f"Текущий максимальный ID: {current_max_id}. Следующий ID: {next_id}")

    target_input = input("Сколько отзывов собрать? (по умолчанию 300): ").strip()
    target_count = int(target_input) if target_input else 300

    existing_urls = load_existing_reviews(CSV_FILE)
    print(f"Загружено {len(existing_urls)} уже собранных отзывов.")

    print(f"Открываю {START_URL}")
    driver.get(START_URL)
    time.sleep(PAGE_LOAD_DELAY)

    print("\n--- Начинаем сбор отзывов ---")
    new_collected = scrape_page(driver, existing_urls, next_id, target_count)

    print(f"\nСбор завершён. Добавлено новых отзывов: {new_collected}")
    print(f"Всего в файле теперь: {len(existing_urls)}")
    driver.quit()


if __name__ == "__main__":
    main()